Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
# @title Imports and constants (run this first)
import sys
import matplotlib.pyplot as plt
import numpy as np
import sympy

sys.set_int_max_str_digits(10000)

# Tiny epsilon for sign checking.
epsilon = sympy.Rational(1, 10**198)

# An uncertainty principle inequality: a Laguerre polynomials approach

We follow the basis construction of [Cohn and Gonçalves (2019)](https://link.springer.com/article/10.1007/s00222-019-00875-4) and refine a few of the constructions therein.

Let $C_4$ be the uncertainty constant discussed in our paper. Setting $C_4 = A_+(1)^2$, the above work by Cohn and Gonçalves produced the upper bound $A_+(1) \leq 0.572990$; using the same search hypothesis, AlphaEvolve improved this to $A_+(1) \leq 0.56708661892$.

Unlike our previous implementation that uses Hermite polynomials as a basis of the construction (inspired by [the previous SOTA](https://www.sciencedirect.com/science/article/pii/S0022247X17301804)), here we use Laguerre polynomials (following Cohn and Gonçalves above). To this end, we need to adjust our verifier, as seen below.

In [ ]:
# @title Solution found by AlphaEvolve

roots = [
  3.6331003,
  5.6714292,
  33.09981679,
  38.849942887744,
  41.1543366,
  43.18733472973041,
  50.98385922,
  58.63890192096693,
  96.0237184433404,
  111.2160645819441,
  118.90258667688155,
  141.44196227075895,
]


def construct_laguerre_combination(root_points: list[float]) -> sympy.Expr:
  """Constructs a linear combination of Laguerre polynomials g(x) with prescribed double roots.

  Constraints:
  - Sets g(0) = 0 and g'(0) = 1.
  - Sets g(z) = 0 and g'(z) = 0 for each z in root_points.

  Args:
    root_points: the list of double roots.

  Returns:
    A sympy expression g(x), as a linear combination of Laguerre polynomials,
    satisfying the above constraints.
  """
  # 1. Setup Parameters
  m = len(root_points)

  # The shape parameter (alpha) determines the specific family of
  # Laguerre polynomials. For the uncertainty principle in 1D,
  # we have alpha = -1/2.
  alpha = sympy.Rational(1, 2) - 1
  x = sympy.symbols('x')

  # 2. Define the polynomial basis: We need 2m + 2 polynomials to satisfy 2m + 2 constraints.
  degrees = np.arange(0, 4 * m + 4, 2)
  basis_polys = [
    sympy.polys.orthopolys.laguerre_poly(
        n=int(d), x=x, alpha=alpha, polys=False
    )
    for d in degrees
  ]
  num_basis = len(basis_polys)
  num_constraints = 2 * m + 2
  # Pre-calculate derivatives of the basis for the matrix construction.
  basis_derivs = [p.diff(x) for p in basis_polys]

  # 3. Initialize the linear system for finding the coefficients.
  matrix = sympy.Matrix.zeros(num_constraints, num_basis)
  targets = sympy.Matrix.zeros(num_constraints, 1)

  # 4. Apply the boundary conditions at x=0: g(0) = 0 and g'(0) = 1.
  targets[1] = 1
  for j in range(num_basis):
    matrix[0, j] = basis_polys[j].subs(x, 0)
    matrix[1, j] = basis_derivs[j].subs(x, 0)

  # 5. Apply the double root conditions at each root z: g(z) = 0, g'(z) = 0.
  for i, val in enumerate(root_points):
    z_i = sympy.Rational(val)
    row_val = 2 * i + 2
    row_der = 2 * i + 3
    for j in range(num_basis):
      matrix[row_val, j] = basis_polys[j].subs(x, z_i)
      matrix[row_der, j] = basis_derivs[j].subs(x, z_i)

  # 6. Solve the system using LUsolve and assemble the final function g(x).
  coeffs = matrix.LUsolve(targets)

  return sum(coeffs[j] * basis_polys[j] for j in range(num_basis))


g_fn = construct_laguerre_combination(roots)

In [ ]:
# @title Verification


def verify_laguerre_construction(
    g_fn: sympy.Expr, expected_double_roots: list[float]
) -> tuple[bool, str, sympy.Expr | None]:
  """Verifies that the constructed function g(x) satisfies these constraints: 1.

  g(0) == 0. 2. g'(0) == 1 (The anchor condition). 3. g(z) == 0 for all z in
  roots. 4. g'(z) == 0 for all z in roots (Double root condition). 5. gq_fn (g
  after factoring out known roots) must have real roots (sign changes).

  Returns:
    (bool, str, sympy.Expr): (True, Success message, gq_fn) or (False, Error
    description, None)
  """
  x = sympy.symbols("x")
  dg_fn = sympy.diff(g_fn, x)

  # 1. Check conditions at x = 0
  # Verify g(0) = 0.
  if not g_fn.subs(x, 0).is_zero:
    return (
        False,
        f"Constraint failed: g(0) is {g_fn.subs(x, 0)}, expected 0.",
        None,
    )

  # Verify g'(0) = 1.
  if dg_fn.subs(x, 0) != 1:
    return (
        False,
        f"Constraint failed: g'(0) is {dg_fn.subs(x, 0)}, expected 1.",
        None,
    )

  # 2. Check double roots at all points of expected_double_roots.
  for z in expected_double_roots:
    z_rational = sympy.Rational(z)
    # Verify g(z) = 0.
    if not g_fn.subs(x, z_rational).is_zero:
      return False, f"Constraint failed: g({z}) is not 0.", None
    # Verify g'(z) = 0.
    if not dg_fn.subs(x, z_rational).is_zero:
      return False, f"Constraint failed: g'({z}) is not 0.", None

  # 3. Factor out the known roots to check for the 'uncertainty' sign change,
  # div = x * product of (x - z_i)^2.
  div = x * sympy.prod(
      [(x - sympy.Rational(z)) ** 2 for z in expected_double_roots]
  )

  try:
    # Compute exact quotient.
    gq_fn = sympy.exquo(g_fn, div)
  except Exception as e:
    return False, f"Polynomial division failed: {str(e)}", None

  # 4. Check for real roots in the quotient (finding the upper bound).
  if not sympy.real_roots(gq_fn, x):
    return (
        False,
        (
            "Verification failed: No sign changes (real roots) found in the"
            " quotient."
        ),
        None,
    )

  return (
      True,
      (
          "All constraints verified: g(0)=0, g'(0)=1, and all double roots are"
          " enforced."
      ),
      gq_fn,
  )


is_valid, message, gq_fn = verify_laguerre_construction(g_fn, roots)
print(message)

if is_valid:
  print("Laguerre polynomial construction is valid.")
else:
  print("Laguerre polynomial construction is invalid!!")

In [ ]:
# @title Compute the upper bound

def compute_upper_bound(gq_fn: sympy.Expr) -> float:
  """Computes the C4 upper bound from the quotient function.

  Assumes gq_fn has already been verified and extracted.
  """
  x = sympy.symbols("x")
  largest_sign_change = sympy.Integer(0)

  for root in sympy.real_roots(gq_fn, x):
    # High precision evaluation
    approx_root = root.eval_rational(n=200)

    # Test points
    val_p = gq_fn.subs(x, approx_root + epsilon)
    val_m = gq_fn.subs(x, approx_root - epsilon)

    # Check for sign change (crossing)
    if (val_p * val_m) < 0:
      if approx_root > largest_sign_change:
        largest_sign_change = approx_root

  # Calculate C4
  c4_bound = float(largest_sign_change) / (2 * np.pi)
  return c4_bound


c4_bound = compute_upper_bound(gq_fn)
a1_bound = float(np.sqrt(c4_bound))
print(
    "AlphaEvolve construction implies the upper bound: A+(1) <="
    f" {repr(a1_bound)} and C4 <= {repr(c4_bound)}."
)

In [ ]:
# @title Visualization
x = sympy.symbols("x")
gn_fn = sympy.lambdify(x, g_fn, modules=["numpy"])
g_fn.subs(x, sympy.Rational(0))

x_vals = np.linspace(-0.1, 4, 200)

plt.figure(figsize=(10, 6))
plt.plot(
    x_vals,
    gn_fn(x_vals),
    label="g(x) = Linear combination of Laguerre Polynomials",
)

plt.axhline(0, color="black", linestyle="--", linewidth=0.5)  # x-axis
plt.xlabel("x")
plt.ylabel("g(x)")
plt.title(
    "Laguerre Polynomial Linear Combination with Specified Roots found by"
    " AlphaEvolve"
)
plt.legend()
plt.grid(True)
plt.show()